# Further Experiment 2 — Small-LLM zero-shot baselines

Where do the three trained models sit against a modern **instruction-tuned LLM that was never trained on Sentiment140**? We prompt a ~3B model (`Qwen2.5-3B-Instruct`) to label tweets zero-shot (no examples) on the same 1,000 held-out tweets, and we try **two prompts**:

1. a plain prompt, and
2. a detailed prompt built from the original Sentiment140 paper's task definition (emoticon distant supervision, 'a personal positive/negative feeling', strictly binary).

**What we noticed:** the LLM lands *below* the trained models (~0.74), and the 'better' paper-informed prompt actually **lowered accuracy** (0.735 -> 0.699) while raising F1 — it fixed the model's tendency to over-call *negative* but overshot into over-calling *positive*. The deeper reason: the LLM judges the *real* sentiment, but the gold labels come from noisy emoticons, so a chunk of its 'errors' are really it disagreeing with the label noise.

> The original experiment ran this through `llama.cpp` for fast batched inference; here we use plain `transformers` so it runs in Colab. Use a GPU runtime.

## Setup

In [ ]:
!pip install -q datasets transformers accelerate scikit-learn

In [ ]:
import re, random, numpy as np, pandas as pd, torch
from datasets import load_dataset, concatenate_datasets
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from transformers import AutoTokenizer, AutoModelForCausalLM
SEED = 42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
def clean_tweet(text):
    text = str(text)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)   # strip URLs
    text = re.sub(r'@\w+', '@user', text)                  # @mentions -> @user
    text = re.sub(r'\s+', ' ', text).strip()               # normalize whitespace
    return text

## Build the same 1,000-tweet evaluation set (balanced)

In [ ]:
ds = load_dataset('contemmcm/sentiment140')['complete']
neg = ds.filter(lambda x: x['label'] == 0).shuffle(seed=SEED).select(range(500))
pos = ds.filter(lambda x: x['label'] == 1).shuffle(seed=SEED).select(range(500))
eval_ds = concatenate_datasets([neg, pos]).shuffle(seed=SEED)
texts = [clean_tweet(t) for t in eval_ds['text']]
labels = np.array(eval_ds['label'])
print('eval set:', len(texts), 'balance:', labels.mean())

## Load the small LLM

In [ ]:
MODEL = 'Qwen/Qwen2.5-3B-Instruct'
tok = AutoTokenizer.from_pretrained(MODEL)
llm = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16,
                                           device_map='auto')
llm.eval()

## Prompting + parsing

We let the model talk, then parse the final `Sentiment: POSITIVE/NEGATIVE` line (with a polarity-word fallback). It must commit — there is no neutral.

In [ ]:
import re
def classify(text, system):
    msgs = [{'role': 'system', 'content': system},
            {'role': 'user', 'content': f'Tweet: {text}'}]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = tok(prompt, return_tensors='pt').to(device)
    out = llm.generate(**ids, max_new_tokens=128, do_sample=False,
                       pad_token_id=tok.eos_token_id)
    txt = tok.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True)
    m = re.findall(r'sentiment\s*[:\-]\s*\**\s*(positive|negative)', txt, re.I)
    if m: return 1 if m[-1].lower() == 'positive' else 0
    w = re.findall(r'\b(positive|negative)\b', txt, re.I)
    return (1 if w[-1].lower() == 'positive' else 0) if w else 0

def run(system, name):
    preds = np.array([classify(t, system) for t in texts])
    acc = accuracy_score(labels, preds); f1 = f1_score(labels, preds)
    tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
    print(f'{name:14s} acc={acc:.4f} f1={f1:.4f} '
          f'pos_rec={tp/(tp+fn):.3f} neg_rec={tn/(tn+fp):.3f} pred_pos={preds.sum()}')
    return acc, f1

### Prompt A — plain zero-shot

In [ ]:
SYS_PLAIN = ('You are a precise sentiment classifier for short tweets. Decide whether '
             'the overall sentiment is POSITIVE or NEGATIVE. Every tweet is one or the '
             'other, never neutral. You may explain briefly, but END your reply with '
             'exactly:\nSentiment: POSITIVE\nor\nSentiment: NEGATIVE')
run(SYS_PLAIN, 'plain')

### Prompt B — paper-informed (no examples, just a richer task definition)

Built from Go, Bhayani & Huang (2009): labels come from emoticon distant supervision, sentiment = 'a personal positive or negative feeling', binary, and the common mistakes to avoid (don't default to negative, don't moralize, playful exaggeration is positive).

In [ ]:
SYS_PAPER = ('You are an expert annotator for the Sentiment140 benchmark. Label the tweet '
             'POSITIVE or NEGATIVE.\n'
             '- Sentiment means a personal positive or negative FEELING of the author, not '
             'whether the facts are good/bad/true/moral.\n'
             '- Labels come from emoticons (a smiley = positive, a frowny = negative); infer '
             'which the author would have used.\n'
             '- Strictly binary, never neutral: commit even on mild/terse tweets.\n'
             '- Do NOT default to negative for short/serious/technical tweets; do not '
             'moralize; playful exaggeration and mock-threats are POSITIVE.\n'
             'End with exactly:\nSentiment: POSITIVE\nor\nSentiment: NEGATIVE')
run(SYS_PAPER, 'paper')

## What we found

On the same 1,000 tweets (our offline `llama.cpp` run, identical method):

| Prompt | Accuracy | Pos-F1 | Pos recall | Neg recall | Predicted positive |
|---|---|---|---|---|---|
| Plain | **0.735** | 0.727 | 0.71 | 0.76 | 471 / 1000 |
| Paper-informed | 0.699 | **0.746** | **0.88** | 0.52 | 683 / 1000 |

And the **three trained models on these exact 1,000 tweets**: DistilBERT 0.810, TF-IDF + LR 0.794, BiLSTM 0.775 — all clearly ahead of the LLM.

- The zero-shot LLM is impressive for **zero training data**, but it loses to fine-tuning on (even noisy) in-distribution labels.
- The paper-informed prompt **moved the operating point** (less negative-bias -> more positive-bias), raising F1 but lowering accuracy — it did not lift the noisy-label ceiling. Prompt wording trades one bias for another.
- Many LLM 'errors' are it reading the genuine sentiment while the gold label reflects an emoticon the words don't match.